In [78]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

In [ ]:
df = pd.read_csv('data/course_lead_scoring.csv')

In [ ]:
df.isna().sum()

In [ ]:
df.dtypes

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns
num_cols = df.select_dtypes(include=['number']).columns

df[categorical_cols] = df[categorical_cols].fillna('NA')
df[num_cols] = df[num_cols].fillna(0.0)

In [ ]:
df.isna().sum()

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [ ]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [ ]:
df_train.nunique()

# ROC AUC feature importance

In [ ]:
num_feats = ['lead_score', 'number_of_courses_viewed', 'interaction_count', 'annual_income']

results = []

for col in num_feats:
    auc = roc_auc_score(y_train, df_train[col])
    if auc < 0.5:
        # print(y_train)
        df_train[col] =  -df_train[col]
        auc = roc_auc_score(y_train, df_train)
    results.append((col, auc))

# Show sorted, best on top
results = sorted(results, key=lambda t: t[1], reverse=True)
results

### Question 1: Which numerical variable (among the following 4) has the highest AUC?
#### Answer is number_of_courses_viewed

# Training the model

In [ ]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[list(categorical_cols) + num_feats].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[list(categorical_cols) + num_feats].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [ ]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict_proba(X_val)[:, 1]

In [ ]:
auc = roc_auc_score(y_val, y_pred)   # y_pred are your probabilities for class 1
print("AUC:", round(auc, 3))

### Question 2: What's the AUC of this model on the validation dataset? (round to 3 digits)
#### Answer is 0.72

# Precision and Recall

In [ ]:
actual_positive = (y_val == 1)
actual_negative = (y_val == 0)

In [ ]:
scores = []
thresholds = np.linspace(0, 1, 101)

for t in thresholds:
    predict_positive = (y_pred >= t)
    predict_negative = (y_pred < t)

    tp = (predict_positive & actual_positive).sum()
    tn = (predict_negative & actual_negative).sum()
    
    fp = (predict_positive & actual_negative).sum()
    fn = (predict_negative & actual_positive).sum()

    # Precision
    p = tp / (tp + fp)

    # Recall
    r = tp / (tp + fn)

    scores.append((t, p, r))

pr_table = pd.DataFrame(scores, columns=['threshold', 'precision', 'recall'])
print(pr_table.head())
print(pr_table.tail())
    

In [ ]:
columns = ['threshold', 'precision', 'recall']
df_scores = pd.DataFrame(scores, columns=columns)

In [ ]:
plt.plot(df_scores.threshold, df_scores['precision'], label='Precision')
plt.plot(df_scores.threshold, df_scores['recall'], label='Recall')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision vs Recall')
plt.legend()

### At which threshold precision and recall curves intersect?
#### Answer: 0.745